# Lab 2 · Land use in hectares

**Day 1 · about 15 minutes · Student notebook**

> **Goal.** Turn a ready-made global land cover map into a table of hectares per class for your province — then discover that two respected products disagree.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## Someone already made the map

You do not have to classify land cover yourself. Several teams have already done it for the
whole planet. Your job today is to **use their map and get a number out of it** — because
"how many hectares of forest?" is the question your job will actually ask.

### Setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

In [ ]:
# Study area: one Thai province. No shapefile upload needed.
PROVINCE = 'Nan'        # <-- change to your own province

aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()

area_ha = aoi.area(maxError=100).divide(1e4).getInfo()
print(f'{PROVINCE}: {area_ha:,.0f} hectares')

### Step 1 — ESA WorldCover

#### 🤖 Prompt card — Land cover map with a proper legend

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Show a land cover map of my province with the correct official colours and a legend.
RESOURCE  Use ESA/WorldCover/v200/2021, band "Map".
OUTLINE   Clip to `aoi`.
MUST      Use the official ESA class colours. Classes are 10 tree, 20 shrub, 30 grass,
          40 crop, 50 built, 60 bare, 70 snow, 80 water, 90 wetland, 95 mangrove, 100 moss.
          Add a readable legend.
PLATFORM  Google Colab with geemap.
TEST      Print the list of class values actually present in my area.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


### Step 2 — turn the map into hectares

This is the step that matters. A picture is not a result; **a number is a result.**

The trick is `ee.Image.pixelArea()`, which gives every pixel its true area in m². Group that
by land cover class, sum it, divide by 10,000, and you have hectares.

#### 🤖 Prompt card — Area per class in hectares

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Calculate how many hectares of each land cover class are in my province.
RESOURCE  Use the WorldCover image already in the variable `wc`.
OUTLINE   Region is `aoi`.
MUST      Use ee.Image.pixelArea() divided by 10000 for hectares.
          Group by the class band with a grouped sum reducer.
          Use scale=100 and maxPixels=1e9 so I do not blow my quota.
          Print a tidy table sorted largest first, with a percentage column.
PLATFORM  Google Colab.
TEST      The class areas must add up to roughly the province area.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> For **Nan** you should see roughly:
> 
> | Class | Hectares | Share |
> |---|---:|---:|
> | Tree cover | 1,033,493 | 84.4% |
> | Grassland | 126,798 | 10.4% |
> | Cropland | 52,639 | 4.3% |
> | Built-up | 7,497 | 0.6% |
> | Water | 3,100 | 0.3% |
> 
> **Total must be close to the province area you printed earlier (1,227,687 ha).** If your total is
> ten times too big or too small, you divided by the wrong number.

### Step 3 — a second opinion

Now do the same thing with a completely different product, **Dynamic World**, and compare.

In [ ]:
dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
      .filterBounds(aoi).filterDate('2024-11-01', '2025-02-28')
      .select('label').mode().clip(aoi))

DW_NAMES = ['water', 'trees', 'grass', 'flooded_vegetation', 'crops',
            'shrub_and_scrub', 'built', 'bare', 'snow_and_ice']

dw_areas = (ee.Image.pixelArea().divide(1e4).addBands(dw)
            .reduceRegion(ee.Reducer.sum().group(groupField=1, groupName='class'),
                          geometry=aoi, scale=100, maxPixels=1e9, bestEffort=True)
            .getInfo())

dw_rows = sorted(dw_areas['groups'], key=lambda g: -g['sum'])
dw_total = sum(g['sum'] for g in dw_rows)
print(f"{'Dynamic World class':<22}{'Hectares':>14}{'Share':>9}")
print('-' * 45)
for g in dw_rows:
    print(f"{DW_NAMES[g['class']]:<22}{g['sum']:>14,.0f}{g['sum']/dw_total*100:>8.1f}%")

wc_tree = next(g['sum'] for g in rows if g['class'] == 10)
dw_tree = next(g['sum'] for g in dw_rows if g['class'] == 1)
print(f"\nWorldCover tree cover : {wc_tree:>12,.0f} ha  ({wc_tree/total*100:.1f}%)")
print(f"Dynamic World trees   : {dw_tree:>12,.0f} ha  ({dw_tree/dw_total*100:.1f}%)")
print(f"Difference            : {abs(dw_tree-wc_tree):>12,.0f} ha")

> ### ✓ Check your answer
>
> ### The two maps disagree — and that is the lesson
> 
> For Nan: WorldCover says **1,033,493 ha** of tree cover (84%). Dynamic World says
> **1,085,511 ha** (88%). That is a **52,000 hectare** gap — an area the size of a small district.
> 
> Neither is lying. They disagree because:
> 
> - they use **different definitions** of "tree" (canopy height? canopy cover percent?),
> - WorldCover is a fixed **2021** snapshot, Dynamic World is a rolling composite of *your* dates,
> - WorldCover has one class for grass and shrub where Dynamic World splits them.
> 
> **What you must take from this:** never report "the forest area of Nan is X ha" full stop.
> Report *"X ha according to ESA WorldCover 2021"*. The product and the year are part of the answer.